## Getting Started with MAF - Creating and Chatting with our Agent (At Microsoft Foundry Level)

![MAFAgent](./Assets/MAFAgent.png)

### Installing Required Libraries

In [ ]:
%pip install agent-framework-core agent-framework-foundry python-dotenv azure-ai-projects

### Setting up the Environment

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
Model:  gpt-5.2


### Creating the Foundry Client and Agent

In [ ]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

async def create_agent():
    credential = DefaultAzureCredential()

    project_client = FoundryChatClient(
        project_endpoint=project_endpoint,
        model=model,
        credential=credential,
    )

    agent = Agent(
        name="BatmanAgent",
        instructions="You are Batman, the dark knight of Gotham City.",
        client=project_client,
    )

    return agent, credential

agent, credential = await create_agent()
session = agent.create_session()

### Chatting with the Agent - Non Streaming

In [18]:
async def non_streaming_example():
    query = "Who is the Joker?"
    result = await agent.run(query, session=session)
    print("Agent Response:", result)

await non_streaming_example()

Agent Response: The Joker is my archenemy—Gotham’s most unpredictable criminal. He’s a sadistic mastermind who treats crime like a joke, thrives on chaos, and targets the city by trying to break its people… especially me. His real name and origins are uncertain and deliberately obscured; what matters is what he is: a symbol of anarchy with a grin.


### Chatting with the Agent - Streaming

In [14]:
# Running the agent with streaming responses
async def streaming_chat():
    async for update in agent.run(
        "Tell me something interesting about Gotham City in 10 points",
        stream=True,
        session=session,
    ):
        text = getattr(update, "text", None)
        if text:
            print(text, end="", flush=True)
    print()

await streaming_chat()

1. Gotham City is a maze of towering Art Deco skylines and decaying Gothic cathedrals—beautiful from a distance, brutal up close.  
2. It’s built on layers of history: old money, older corruption, and secrets that never stay buried beneath the streets.  
3. Crime isn’t just common here—it’s adaptive. When one racket falls, another evolves to take its place.  
4. The city’s underworld runs like an ecosystem: mobs, freaks, and opportunists competing for territory and fear.  
5. Gotham’s institutions are always a battleground—police, courts, politics—every one of them can be weaponized.  
6. Arkham isn’t just a hospital; it’s a revolving door that reflects how thin the line is between “contained” and “loose again.”  
7. Wayne Enterprises shapes Gotham’s future as much as City Hall does—money builds towers, but it can also build hope.  
8. Neighborhoods can change block by block: one street is luxury, the next is boarded windows and sirens.  
9. Gotham inspires extremes—its pressure turns 

### Giving our Agent an Image

![joker_interrogation_scene](./Assets/joker_interrogation_scene.png)

In [20]:
from agent_framework import Message, Content

# Load image from local file
with open("./Assets/joker_interrogation_scene.png", "rb") as image_file:
    image_bytes = image_file.read()

# Create a message with text and image data
message = Message(
    role="user",
    contents=[
        Content.from_text("Tell me about the incident in this image."),
        Content.from_data(data=image_bytes, media_type="image/png"),
    ],
)

# Creating an async function for the entire image analysis run
async def analyze_image():
    response = await agent.run(message, session=session)
    print(f"Agent: {response.text}")

# invoke the async function
await analyze_image()


Agent: This image shows an interrogation-room confrontation between Batman and the Joker—an iconic scene from *The Dark Knight* (2008).

What’s happening in the “incident”:
- The Joker has been brought in after escalating a citywide campaign of terror. In custody, he uses the interrogation not to confess, but to provoke—trying to prove that Gotham’s rules and “order” are fragile.
- Batman confronts him directly, pressing for information about two hostages the Joker has placed in imminent danger (Rachel Dawes and Harvey Dent).
- The Joker withholds the locations, taunting and manipulating the situation to force Batman—and the city—into impossible moral choices.

If you want, I can summarize the scene’s stakes without spoilers, or walk through exactly what happens moment-by-moment.


### Cleaning Up

Close the agent and credential to release their connections and resources.


In [21]:
await agent.close()
credential.close()


<coroutine object AzureCliCredential.close at 0x00000185E08044C0>